## Tutorial

https://www.kaggle.com/code/vitouphy/phoneme-recognition-with-wav2vec2

### DEMO DATA 

https://www.kaggle.com/datasets/mfekadu/darpa-timit-acousticphonetic-continuous-speech/

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version torch was built with:", torch.version.cuda)

Torch version: 2.10.0+cu126
CUDA available: True
CUDA version torch was built with: 12.6


## Creating .env file with this IDE

In [2]:
# # TESTING ROOTS
# import pandas as pd
# ROOT = os.path.join("..", "..", "..",)
# fpath = os.path.join(ROOT, "tst.csv")
# d = pd.DataFrame()
# d.to_csv(fpath)

In [3]:
# # Creating .env file

# import os

# env_content = """07_FR_phone_TokenType_READ=####### --- Token ----#######
# """
# #____________________________________________
# ROOT = os.path.join("..", "..", "..",)
# fpath = os.path.join(ROOT, ".env")
# with open(fpath, "w") as f:
#     f.write(env_content)

## Config HF Cache

In [3]:
import os

class cfg: 

    # to store HF pre-trained models weights and configs
    HF_CACHE_ROOT = os.path.join("..", "..", "..",
                                 "data",
                                 "05_cache", 
                                 "HF"
                                )



    # to store HF pre-trained models weights and configs
    HF_FINETUNE_ROOT = os.path.join("..", "..", "..",
                                    "data",
                                    "06_fine_tune",
                                    "01_tuto",
                                    "01_hug_llm",
                                    "ch03",
                                   )


    HF_FINETUNED_MODEL_SAVE_ROOT = os.path.join(HF_CACHE_ROOT,
                                                "FR_finetuned",
                                               )

## HF Cache management

https://huggingface.co/docs/datasets/en/cache

In [4]:
print("HF_HOME:", os.environ.get("HF_HOME"))
os.environ["HF_HOME"] = cfg.HF_CACHE_ROOT
print("HF_HOME:", os.environ.get("HF_HOME"))

HF_HOME: None
HF_HOME: ../../../data/05_cache/HF


In [5]:
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))
os.environ["HF_HUB_CACHE"] = cfg.HF_CACHE_ROOT
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HUB_CACHE: None
HF_HUB_CACHE: ../../../data/05_cache/HF


## Import libraries

In [105]:
import io
import sys
import random
import json



#_________
import torch
import torchaudio

#__________
import librosa

#__________
# HF stack
import transformers
from transformers import pipeline

import datasets 
from datasets import load_dataset 
# from datasets import load_metric # deprecated, Now I'm using evaluate
from datasets import Audio as Audio_ds # the instances generated from load_dataset use under the hood Audio to decode (Audio use Torchcoced).

from datasets import Dataset

import evaluate

#_________
import numpy as np
import pandas as pd 

#_________
from dotenv import load_dotenv

#_________
from tqdm.auto import tqdm

#_________
from IPython.display import Audio
from IPython.display import display, HTML

In [7]:
transformers.__version__

'5.1.0'

In [8]:
torch.__version__

'2.10.0+cu126'

In [9]:
datasets.__version__

'4.5.0'

In [10]:
torch.cuda.is_available()

True

In [11]:
torch.cuda.device_count()

1

In [56]:
librosa.__version__

'0.11.0'

## Utilities

### safe_unzip()

In [13]:
import zipfile
import os

def safe_unzip(zip_path, extract_path=None):
    try:
        # Validate zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Set extract path
        if extract_path is None:
            extract_path = os.path.splitext(zip_path)[0]
        
        # Create directory if needed
        os.makedirs(extract_path, exist_ok=True)
        
        # Extract
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Test if zip is valid
            bad_file = zip_ref.testzip()
            if bad_file:
                raise zipfile.BadZipFile(f"Corrupted file in zip: {bad_file}")
            
            zip_ref.extractall(extract_path)
            print(f"Successfully extracted to: {extract_path}")
            
    except FileNotFoundError as e:
        print(f"Error: {e}")
    except zipfile.BadZipFile as e:
        print(f"Error: Invalid or corrupted zip file - {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

## Service Token Authentication

In [12]:
# Verify token is loaded
load_dotenv()

HF_TOKEN_READ = os.getenv("07_FR_phone_TokenType_READ")
print(f"Token loaded: {'Yes' if HF_TOKEN_READ else 'No'}")

Token loaded: Yes


## Unzip dataset

In [14]:
TIMIT_KAGGLE_ROOT = os.path.join("..", "..", "..",
                                 'data', 
                                 '06_timit_kaggle',
                                ) 

zip_fname = 'timit_kaggle.zip'

zip_fpath = os.path.join(TIMIT_KAGGLE_ROOT, zip_fname)
zip_extract_path = TIMIT_KAGGLE_ROOT

safe_unzip(zip_path=zip_fpath, extract_path=zip_extract_path)
print(f"zip file has been extrated in this folder: \n {zip_extract_path}")

Successfully extracted to: ../../../data/06_timit_kaggle
zip file has been extrated in this folder: 
 ../../../data/06_timit_kaggle


## Paths

In [16]:
TIMIT_KAGGLE_ROOT = os.path.join("..", "..", "..",
                                 'data', 
                                 '06_timit_kaggle',
                                )
#=========================================================
data_path = os.path.join(TIMIT_KAGGLE_ROOT, 'data') 


## Load metadata

In [19]:
df_train = pd.read_csv(os.path.join(TIMIT_KAGGLE_ROOT, 'train_data.csv'))
df_train

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TRAIN,DR4,MMDM0,SI681.WAV.wav,TRAIN/DR4/MMDM0/SI681.WAV.wav,TRAIN\\DR4\\MMDM0\\SI681.WAV.wav,True,True,False,False,False
1,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
2,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False
3,4.0,TRAIN,DR4,MMDM0,SX321.PHN,TRAIN/DR4/MMDM0/SX321.PHN,TRAIN\\DR4\\MMDM0\\SX321.PHN,False,False,False,True,False
4,5.0,TRAIN,DR4,MMDM0,SX321.WRD,TRAIN/DR4/MMDM0/SX321.WRD,TRAIN\\DR4\\MMDM0\\SX321.WRD,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
31673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31674,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_test = pd.read_csv(os.path.join(TIMIT_KAGGLE_ROOT, 'test_data.csv'))
df_test

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TEST,DR4,MGMM0,SX139.WAV,TEST/DR4/MGMM0/SX139.WAV,TEST\\DR4\\MGMM0\\SX139.WAV,False,True,False,False,False
1,2.0,TEST,DR4,MGMM0,SX139.WAV.wav,TEST/DR4/MGMM0/SX139.WAV.wav,TEST\\DR4\\MGMM0\\SX139.WAV.wav,True,True,False,False,False
2,3.0,TEST,DR4,MGMM0,SX139.TXT,TEST/DR4/MGMM0/SX139.TXT,TEST\\DR4\\MGMM0\\SX139.TXT,False,False,False,False,True
3,4.0,TEST,DR4,MGMM0,SI499.WRD,TEST/DR4/MGMM0/SI499.WRD,TEST\\DR4\\MGMM0\\SI499.WRD,False,False,True,False,False
4,5.0,TEST,DR4,MGMM0,SX319.WRD,TEST/DR4/MGMM0/SX319.WRD,TEST\\DR4\\MGMM0\\SX319.WRD,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
31673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31674,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df = pd.concat([df_train, df_test])
df

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TRAIN,DR4,MMDM0,SI681.WAV.wav,TRAIN/DR4/MMDM0/SI681.WAV.wav,TRAIN\\DR4\\MMDM0\\SI681.WAV.wav,True,True,False,False,False
1,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
2,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False
3,4.0,TRAIN,DR4,MMDM0,SX321.PHN,TRAIN/DR4/MMDM0/SX321.PHN,TRAIN\\DR4\\MMDM0\\SX321.PHN,False,False,False,True,False
4,5.0,TRAIN,DR4,MMDM0,SX321.WRD,TRAIN/DR4/MMDM0/SX321.WRD,TRAIN\\DR4\\MMDM0\\SX321.WRD,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
31673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31674,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
c1 = df['is_converted_audio'] == False
df = df[c1].reset_index(drop=True)
df

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
1,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False
2,4.0,TRAIN,DR4,MMDM0,SX321.PHN,TRAIN/DR4/MMDM0/SX321.PHN,TRAIN\\DR4\\MMDM0\\SX321.PHN,False,False,False,True,False
3,5.0,TRAIN,DR4,MMDM0,SX321.WRD,TRAIN/DR4/MMDM0/SX321.WRD,TRAIN\\DR4\\MMDM0\\SX321.WRD,False,False,True,False,False
4,6.0,TRAIN,DR4,MMDM0,SI681.TXT,TRAIN/DR4/MMDM0/SI681.TXT,TRAIN\\DR4\\MMDM0\\SI681.TXT,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
25195,8395.0,TEST,DR8,MPAM0,SX19.WAV,TEST/DR8/MPAM0/SX19.WAV,TEST\\DR8\\MPAM0\\SX19.WAV,False,True,False,False,False
25196,8396.0,TEST,DR8,MPAM0,SX109.TXT,TEST/DR8/MPAM0/SX109.TXT,TEST\\DR8\\MPAM0\\SX109.TXT,False,False,False,False,True
25197,8398.0,TEST,DR8,MPAM0,SX289.WRD,TEST/DR8/MPAM0/SX289.WRD,TEST\\DR8\\MPAM0\\SX289.WRD,False,False,True,False,False
25198,8399.0,TEST,DR8,MPAM0,SX109.WAV,TEST/DR8/MPAM0/SX109.WAV,TEST\\DR8\\MPAM0\\SX109.WAV,False,True,False,False,False


## Merging files for same subID

timit dataset come with in files, not organize in train test, this will help to process the data when I pay for it

In [42]:
%%time
data = {}

for idx, row in tqdm(df.iterrows()):

    # meta file with linux sys path sep to audio file. 
    #  Each audio file name is related to subID
    #  the root folder that a "data" folder with all wav file, phonetic, word files. 
    path = row['path_from_data_dir'] 

    # Extracting from path of wav the subId path. 
    entry_id = path.split('.')[0]

    #__________________________________
    # Creating empty dictionary to story subID info.
    if entry_id not in data:
        data[entry_id] = {}

    #__________________________________
    # data_path is the root in the system to "data" folder in TIMIT kaggle root folder.
    if row['is_audio'] is True:
        data[entry_id]['audio_file'] = os.path.join(data_path, path)

    elif row['is_word_file'] is True:
        data[entry_id]['word_file'] = os.path.join(data_path, path)

    elif row['is_phonetic_file'] is True:
        data[entry_id]['phonetic_file'] = os.path.join(data_path, path)


    #__________________________________
    
    # break

0it [00:00, ?it/s]

CPU times: user 950 ms, sys: 0 ns, total: 950 ms
Wall time: 949 ms


In [46]:
len(data.keys()) # there are 6300 entryID which I think is subID, at this point I dont know if each subID has all three docs, .wav, .phn and .wrd 

6300

### Extracting subID with 3 files

In [48]:
keys = [key for key in data.keys() if len(data[key]) == 3]
len(keys)

3360

** it seems that out of the 6300, only 3360 has the three files 

## Split data into train,val,test

In [92]:
random.Random(101).shuffle(keys)

num_train = int(len(keys) * 0.8)
num_valid = int(len(keys) * 0.1)
num_test = len(keys) - num_train - num_valid

train_keys = keys[:num_train]
valid_keys = keys[num_train:num_train + num_valid]
test_keys = keys[-num_test:]

In [93]:
print(f"train_keys {len(train_keys )}")
print(f"valid_keys {len(valid_keys )}")
print(f"test_keys {len(test_keys )}")

train_keys 2688
valid_keys 336
test_keys 336


In [94]:
# form the data dictionary and with the keys split into train, val and test, organize data for each set 
train = { key:data[key] for key in train_keys }
valid = { key:data[key] for key in valid_keys }
test  = { key:data[key] for key in test_keys }

## How many hours each set

### get_durations()

In [61]:
def get_durations(dict_data):

    """
    Parameters: 
    ----------
        dict_data: dict_data is the train, val, test or global data with subid and 
                    each subject id has a dictionary with 3 keys to point to 
                    .wav, .phn, or .wrd. 
                

    """
    
    total_durations = 0

    for entry in dict_data.values():
        audio_data, _ = librosa.load(entry['audio_file'], sr=16_000)
        duration = len(audio_data) / 16_000 # seconds
        total_durations += duration

    return int(total_durations)

In [95]:
%%time
print(f"Duration of Train: {get_durations(train) // 60} minutes")
print(f"Duration of Valid: {get_durations(valid) // 60} minutes")
print(f"Duration of Test : {get_durations(test) // 60} minutes\n")

Duration of Train: 137 minutes
Duration of Valid: 17 minutes
Duration of Test : 17 minutes

CPU times: user 913 ms, sys: 130 ms, total: 1.04 s
Wall time: 2.41 s


## SAVE the dictionary as json files 

I create a dictionary for each subID with the file path pointers for .wav, .phn and .wrd, I will store them for fine-tuning 

In [72]:
custom_split_data_path = os.path.join(TIMIT_KAGGLE_ROOT, "01_custom_split_meta")

os.makedirs(custom_split_data_path, exist_ok=True)
custom_split_data_path

'../../../data/06_timit_kaggle/01_custom_split_meta'

In [75]:
fname = 'custom_train.json'
fpath = os.path.join(custom_split_data_path,fname )
with open(fpath, "w") as f:
    json.dump(train, f)

In [76]:
fname = 'custom_valid.json'
fpath = os.path.join(custom_split_data_path,fname )
with open(fpath, "w") as f:
    json.dump(valid, f)

In [77]:
fname = 'custom_test.json'
fpath = os.path.join(custom_split_data_path,fname )
with open(fpath, "w") as f:
    json.dump(test, f)

## Conver to HF dataformat

### Proto

In [96]:
for key, value in train.items(): 

    break

In [97]:
value

{'audio_file': '../../../data/06_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.WAV',
 'phonetic_file': '../../../data/06_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.PHN',
 'word_file': '../../../data/06_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.WRD'}

In [98]:
value['audio_file']

'../../../data/06_timit_kaggle/data/TRAIN/DR3/MILB0/SI2163.WAV'

In [99]:
key

'TRAIN/DR3/MILB0/SI2163'

### convert_to_feature_dict()

In [100]:
def convert_to_feature_dict(data_dict):
    # convert each feature into a list instead
    audio_files = []
    word_files = []
    phonetic_files = []
    for key, value in data_dict.items():
        audio_files.append(value['audio_file'])
        word_files.append(value['word_file'])
        phonetic_files.append(value['phonetic_file'])
    
    return {
        'audio_file': audio_files,
        'word_file': word_files,
        'phonetic_file': phonetic_files
    }

In [101]:
'''
Previously dictonary train, valid and test has this structure 
    trian = {'subId_01': {'audio_file': path_str, 'word_file': path_str, 'phonetic_file': path_str}, 
             'subId_02': {'audio_file': path_str, 'word_file': path_str, 'phonetic_file': path_str}, 
             ...
             }

    with conver_to_feature_dict() the subID will be remove, this is to be in the HF dataset style. 
    trian = { 'audio_file': list with all path of audio file, 
              'word_file': list with all path of .wrd files, 
              'phonetic_file': list with all path of .phn files, 
            }
        now, subID does not appear, but same idx in these three list refer to the same sID. 

'''



train = convert_to_feature_dict(train)
valid = convert_to_feature_dict(valid)
test  = convert_to_feature_dict(test)

In [103]:
train.keys()

dict_keys(['audio_file', 'word_file', 'phonetic_file'])

### transform to dataset HF style

In [106]:
train_dataset = Dataset.from_dict(train)
valid_dataset = Dataset.from_dict(valid)
test_dataset = Dataset.from_dict(test)

In [107]:
train_dataset

Dataset({
    features: ['audio_file', 'word_file', 'phonetic_file'],
    num_rows: 2688
})

## Read files for phonetics

### read_text_file()

In [114]:
def read_text_file(filepath):

    '''
    Description
    -----------

        =================
        Phonetic info
        this is the structure of a .phn file 
            0 3050 h#
            3050 4559 sh
            4559 5723 ix
            5723 6642 hv
            6642 8772 eh
            8772 9190 dcl
            9190 10337 jh

            as you can see it is seperator by space, it looks like time points (for sr=16000)
                and the last [-1] column is the one with the Arphable phonetic symbole. 


        =================
        word info
        similar happens with .wrd file 
            3050 5723 she
            5723 10337 had
            9190 11517 your
            11517 16334 dark
            16334 21199 suit
            21199 22560 in
            22560 28064 greasy
            28064 33360 wash
            33754 37556 water
            37556 40313 all
            40313 44586 year
        as you can see it is seperator by space, it looks like time points (for sr=16000)
            and the last [-1] column per line is the one that contains the word. 
    
    '''

    
    with open(filepath) as f:
        tokens = [line.split()[-1] for line in f]
        return " ".join(tokens)

In [111]:
item_path = test_dataset['phonetic_file'][0]
item_path

'../../../data/06_timit_kaggle/data/TEST/DR7/MNLS0/SI1610.PHN'

In [112]:
read_text_file(filepath=item_path)

'h# f er ah n q ih n s tcl t en tcl dh iy ow l dcl d ae n tcl f eh l tcl s ah m th iy ng ih n dcl d iy f ay nx ax bcl b el f l ae sh th r uw hv axr s epi m ay l h#'

### prepare_text_data() 

In [115]:
def prepare_text_data(item):

    """
    Description:
    ------------
        for mapping the dataset object. 
            HF Dataset object hast the method .map() which will loop for each item in the feature list value. 

    """
    
    item['text'] = read_text_file(item['word_file'])
    item['phonetic'] = read_text_file(item['phonetic_file'])
    return item

### Mapping Dataset object HF

In [116]:
train_dataset

Dataset({
    features: ['audio_file', 'word_file', 'phonetic_file'],
    num_rows: 2688
})

In [117]:
train_dataset = (train_dataset
                 .map(prepare_text_data)
                 .remove_columns(["word_file", "phonetic_file"]))
train_dataset 

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

Dataset({
    features: ['audio_file', 'text', 'phonetic'],
    num_rows: 2688
})

In [118]:
valid_dataset = (valid_dataset
                 .map(prepare_text_data)
                 .remove_columns(["word_file", "phonetic_file"]))
valid_dataset

Map:   0%|          | 0/336 [00:00<?, ? examples/s]

Dataset({
    features: ['audio_file', 'text', 'phonetic'],
    num_rows: 336
})

In [119]:
test_dataset  = (test_dataset
                 .map(prepare_text_data)
                 .remove_columns(["word_file", "phonetic_file"]))
test_dataset

Map:   0%|          | 0/336 [00:00<?, ? examples/s]

Dataset({
    features: ['audio_file', 'text', 'phonetic'],
    num_rows: 336
})

## Standarize Phonetics 

In [120]:
train_dataset['phonetic'][0]

'h# q aa r y ux pcl p aa z ax dx ix v y ix l bcl b iy ao l r ay tcl b ay y axr s eh l f h#'

In [127]:
train_phonetics = [phone for x in train_dataset for phone in x['phonetic'].split()]
train_phonetics.sort()
len(train_phonetics) 

102893

In [128]:
print(set(train_phonetics))

{'aa', 'en', 'el', 'iy', 'dh', 's', 'ih', 'ah', 'z', 'y', 'hv', 'f', 'pcl', 'pau', 'zh', 'uw', 'ay', 'ae', 'ax-h', 'eng', 'p', 'n', 'ax', 'ng', 'nx', 'sh', 'w', 'tcl', 't', 'v', 'd', 'g', 'ow', 'aw', 'hh', 'bcl', 'eh', 'ix', 'k', 'l', 'er', 'dx', 'dcl', 'm', 'r', 'ao', 'epi', 'q', 'ux', 'em', 'kcl', 'h#', 'axr', 'jh', 'oy', 'th', 'ch', 'ey', 'gcl', 'b', 'uh'}


In [126]:
len(set(train_phonetics))

61

**Timit has 61 phonetic sounds**
- Each dataset has different phonetics system

In [ ]:
phon61_map39 = {
    'iy':'iy',  'ih':'ih',   'eh':'eh',  'ae':'ae',    'ix':'ih',  'ax':'ah',   'ah':'ah',  'uw':'uw',
    'ux':'uw',  'uh':'uh',   'ao':'aa',  'aa':'aa',    'ey':'ey',  'ay':'ay',   'oy':'oy',  'aw':'aw',
    'ow':'ow',  'l':'l',     'el':'l',  'r':'r',      'y':'y',    'w':'w',     'er':'er',  'axr':'er',
    'm':'m',    'em':'m',     'n':'n',    'nx':'n',     'en':'n',  'ng':'ng',   'eng':'ng', 'ch':'ch',
    'jh':'jh',  'dh':'dh',   'b':'b',    'd':'d',      'dx':'dx',  'g':'g',     'p':'p',    't':'t',
    'k':'k',    'z':'z',     'zh':'sh',  'v':'v',      'f':'f',    'th':'th',   's':'s',    'sh':'sh',
    'hh':'hh',  'hv':'hh',   'pcl':'h#', 'tcl':'h#', 'kcl':'h#', 'qcl':'h#','bcl':'h#','dcl':'h#',
    'gcl':'h#','h#':'h#',  '#h':'h#',  'pau':'h#', 'epi': 'h#','nx':'n',   'ax-h':'ah','q':'h#' 
}